# 01 - Building the dataset

Scenario: a 9-person customer support team starts using an AI copilot in week 6. The copilot can resolve simple, routine tickets on its own, like password resets or "where is my order" questions. Leadership sees this and raises the team's productivity target, expecting an easy win.

What actually happens is that the AI mostly takes away the easy tickets. So the tickets left for humans are, on average, harder than before. The target doesn't account for that at all.

Here's the 12-week story I'm modeling:

Weeks 1-5 (baseline): normal work, original targets. I use these 5 weeks to figure out what's "normal" for this team.

Week 6 (transition): the AI copilot launches and the target goes up right away. The ticket mix starts changing this same week, that part is immediate. But people's wellbeing and quality don't react instantly, those effects take about a week to show up. So I gave week 6 its own label instead of lumping it in with baseline or with the rough weeks that follow.

Weeks 7-9 (risky lever): the team is under target pressure with no extra support. Quality, wellbeing, and workload balance all get worse.

Weeks 10-12 (sustainable lever): the manager steps in (more staffing, training, a renegotiated target). Quality, wellbeing, and workload balance partly recover. This does not mean the team gets back to its old throughput, more on that later.

One made-up event: an agent leaves in week 11, after the fix has already started. I did this on purpose, to show that the damage from weeks 7-9 doesn't just undo itself the moment a manager steps in.

What this notebook saves (all in the data/ folder):
1. team_weekly.csv - one row per week, every metric as its own column
2. team_weekly_long.csv - one row per week and metric, for flexible charting
3. team_weekly_status.csv - one row per week with the Green/Amber/Red/Unclear call, why it got that call, and the retention indicator

team_phase_summary.csv gets built in the second notebook, 02_sql_aggregation.ipynb.

One design rule I kept throughout: nothing here is ever stored at the individual-agent level, only team totals and a team headcount. That keeps this on the safe side of the EU AI Act's rules about systems that monitor individual workers (Annex III).

## Step 0 - settings I'm locking in before generating anything

The random seed makes the "random" numbers come out the same every time I run this, so it's reproducible.

I'm also locking in the four warning thresholds here, before looking at any generated data, and they don't change later no matter what the data looks like:

- Complexity elevated: at least 10 percentage points above baseline
- Quality deteriorating: CSAT at least 5 percentage points below baseline
- Wellbeing deteriorating: at least 1.0 below baseline on a 0-10 scale
- Workload unfairness elevated: at least 5 percentage points above baseline

These are rules I made up for this project, the kind of number a real ops manager might say out loud. They're not statistically validated limits or an industry standard.

In [ ]:
import numpy as np
import pandas as pd
import os

COMPLEXITY_ELEVATED_PP = 10
QUALITY_DROP_PP        = 5
WELLBEING_DROP         = 1.0
WORKLOAD_ELEVATED_PP   = 5

RNG = np.random.default_rng(42)

WEEKS = list(range(1, 13))
AI_LAUNCH_WEEK = 6
INTERVENTION_WEEK = 10
TURNOVER_WEEK = 11

ACTIVE_AGENTS = 9     # held constant for all 12 weeks, see note below
WORKING_DAYS = 5
WORKING_MINUTES_PER_DAY = 360   # 6 of 8 hours actually spent handling tickets

os.makedirs("data", exist_ok=True)

Why I kept active_agents fixed at 9 the whole time: the one agent who leaves in week 11 is just an event, not something that changes the math. If staffing also changed in week 11, it would add a second moving part right when I'm trying to show the lagging effect of the manager's intervention, and that would just make everything harder to read. The point of this project is how AI changes the work and how management responds to it, not a staffing story.

## Step 1 - label each week with its phase

Week 6 gets its own "transition" label, separate from both baseline and "risky lever".

It's not baseline, because the ticket mix (the structural part) already starts shifting that week.

It's not "risky lever" yet either, because the human effects, quality and wellbeing, haven't caught up yet. Those lag by about a week (see Step 6).

In the 3-phase summary table I build in the next notebook, I just leave the transition week out. It's a one-week pivot, not one of the three real phases.

In [ ]:
def phase_of(week: int) -> str:
    if week <= 5:
        return "baseline"
    elif week == AI_LAUNCH_WEEK:
        return "transition"
    elif week < INTERVENTION_WEEK:
        return "risky_lever"
    else:
        return "sustainable_lever"

phases = [phase_of(w) for w in WEEKS]
PHASE_ORDER = {"baseline": 1, "transition": 1.5, "risky_lever": 2, "sustainable_lever": 3}
PHASE_LABEL = {"baseline": "Baseline", "transition": "Transition",
               "risky_lever": "Risky lever", "sustainable_lever": "Sustainable lever"}
list(zip(WEEKS, phases))

## Step 2 - ticket volume and composition, the actual mechanism

A scaling note first: an earlier draft of this dataset used a small weekly ticket volume, around 220 tickets a week for the whole team. That turned into a problem once I tried to build a real capacity model on top of it (Step 5). At 220 tickets/week, a 9-person team working 6 hours a day would only be using about 16% of its available time, nowhere close to actually being stretched, no matter how much harder the tickets got. For the capacity math to mean anything, baseline demand needs to use a believable share of the team's time. So I raised the baseline ticket volume to around 1,150 a week, about 25-27 tickets per agent per day, which puts baseline utilization at a realistic 80-90%. That's a team that's busy but not overwhelmed, which is what baseline should look like before this story even starts. All the proportions (how much is complex, how fast the AI ramps up) stayed the same as before, I only rescaled the raw ticket count.

The mechanism itself:
1. Every week, some tickets are naturally simple and some are naturally complex. That split comes from the customer base (about 25% complex) and has nothing to do with the AI.
2. Starting week 6, the AI copilot resolves a growing share of the simple tickets on its own. It ramps up over a few weeks rather than jumping straight to full capability.
3. Whatever simple tickets the AI doesn't take, plus every complex ticket, goes to the human team.
4. Because the AI only ever removes simple tickets, the share of the human team's queue that's high-complexity climbs. Not because there are more complex tickets, but because there are fewer simple ones left to balance them out.

In [ ]:
# tickets coming in each week. not affected by the AI copilot, customer
# demand doesn't change just because the team added a tool
tickets_received = np.round(RNG.normal(1150, 55, 12)).astype(int)

# share of all incoming tickets that are naturally complex. comes from the
# customer base, stays roughly constant, just a bit of noise
complex_share_true = np.clip(RNG.normal(0.25, 0.015, 12), 0.18, 0.32)
complex_tickets = np.round(tickets_received * complex_share_true).astype(int)
simple_tickets = tickets_received - complex_tickets

# how much of the simple tickets the AI can resolve on its own, week by
# week. 0 before launch, ramps up starting week 6, then levels off
ai_rate_story = [0,0,0,0,0, 0.30,0.50,0.62,0.66,0.66,0.66,0.66]
ai_rate_noise = RNG.normal(0, 0.03, 12)
# only add noise from week 6 onward. if I add noise to a week where the
# rate should be exactly 0 (weeks 1-5), it can clip back up to a small
# positive number, which would make it look like the AI resolved a few
# tickets before it even launched. weeks 1-5 need to be a true, exact
# pre-AI baseline, so no noise touches them.
ai_rate = np.array(ai_rate_story, dtype=float)
ai_rate[5:] = np.clip(ai_rate[5:] + ai_rate_noise[5:], 0, 0.85)
ai_auto_resolved = np.round(simple_tickets * ai_rate).astype(int)

# what's left for the human team: simple tickets the AI didn't take, plus
# every complex ticket (the AI never touches those)
human_simple_tickets = simple_tickets - ai_auto_resolved
human_tickets_received = human_simple_tickets + complex_tickets

# the number that actually matters: what share of the human team's queue
# is now high-complexity
high_complexity_share_pct = complex_tickets / human_tickets_received * 100

pd.DataFrame({
    "week": WEEKS, "received": tickets_received, "ai_auto_resolved": ai_auto_resolved,
    "human_received": human_tickets_received, "complex_tickets": complex_tickets,
    "pct_high_complexity": np.round(high_complexity_share_pct, 1),
})

Sanity check built in here: by construction, tickets_received always equals ai_auto_resolved plus human_tickets_received, for every single week. No ticket gets created or destroyed by this step. I check this explicitly near the end of the notebook.

## Step 3 - baseline for the complexity share, weeks 1-5 only

The baseline is just "what was normal for this team before anything changed", calculated from the actual weeks 1-5 data, not typed in by hand. I left week 6 out of this calculation on purpose. Even though the human-response metrics are still lagged that week, the ticket composition itself already starts shifting in week 6, so including it would mean averaging in a week that's no longer "normal."

Every later week gets compared back to this baseline, in percentage points, to get a pressure value.

In [ ]:
baseline_complexity_pct = round(high_complexity_share_pct[:5].mean(), 2)
pressure_pp = high_complexity_share_pct - baseline_complexity_pct   # immediate, this-week value

print("Baseline complexity share (weeks 1-5 only):", baseline_complexity_pct, "%")
pd.DataFrame({"week": WEEKS, "pressure_pp": np.round(pressure_pp, 1)})

## Step 4 - handle time, an immediate and mechanical effect of complexity

This one works differently from the human-response metrics in Step 7. Average handle time doesn't need a week to catch up, if the tickets left in the queue are harder this week, they mechanically take longer this week. So avg_handle_time_min responds to this week's pressure value, not a lagged one.

How much it responds matters a lot here, and I didn't pick the number randomly. Since the AI removes roughly half the ticket volume, handle time has to rise by close to double before the team's total workload (in minutes) even gets back to where it was at baseline. A small increase in handle time just isn't enough to outweigh that much less total volume. I check this explicitly in the math section near the end.

In [ ]:
avg_handle_time_min = np.clip(
    np.round(12 + 0.56 * pressure_pp + RNG.normal(0, 0.5, 12), 2), 6, None
)
pd.DataFrame({"week": WEEKS, "avg_handle_time_min": avg_handle_time_min})

## Step 5 - capacity, resolution, and real backlog carryover

This is the part that makes the dataset an actual ticket-flow model instead of a bunch of separately-generated curves that just happen to correlate.

Capacity comes straight from handle time, nothing else:

capacity_per_agent_day = working minutes per day / avg_handle_time_min
weekly_capacity = capacity_per_agent_day times active_agents times working_days

Resolution and backlog are computed as a real weekly carryover, one week feeding into the next:

available = human_tickets_received this week + backlog from last week
human_tickets_resolved = the smaller of available or weekly_capacity (with a small efficiency wobble)
backlog this week = available minus resolved

If the team can get through everything, backlog stays near zero. If demand plus leftover backlog is more than the team can handle that week, the rest carries over.

One rule I enforce strictly: human_tickets_resolved can never go above weekly_capacity. The efficiency noise is only allowed to make a week slightly worse than perfect (as low as 93% of capacity), never better than 100%. A random noise draw shouldn't be able to invent extra working minutes that don't exist. I check this explicitly near the end of the notebook.

In [ ]:
capacity_per_agent_day = WORKING_MINUTES_PER_DAY / avg_handle_time_min
weekly_capacity = capacity_per_agent_day * ACTIVE_AGENTS * WORKING_DAYS

backlog_prev = 60.0   # a small, realistic starting backlog before week 1 (not zero)
human_tickets_resolved, backlog_tickets = [], []
capacity_load_pct, actual_utilization_pct = [], []

for i in range(12):
    available = human_tickets_received[i] + backlog_prev
    # efficiency noise is capped at 1.0 on purpose. a good week can make the
    # team resolve exactly its theoretical capacity, but noise should never
    # let them resolve more than that, since that would mean extra working
    # minutes appeared out of nowhere. noise can only make things slightly
    # worse than perfect, never better.
    efficiency_wobble = np.clip(RNG.normal(1.0, 0.02), 0.93, 1.0)
    resolved = max(0, round(min(available, weekly_capacity[i]) * efficiency_wobble))
    backlog_now = max(0, round(available - resolved))

    # Two different, deliberately separate readings of "how loaded is the team":
    #   capacity_load_pct    = how much work is WAITING to be done, as a share
    #                           of theoretical capacity (demand-side pressure)
    #   actual_utilization_pct = how much of that capacity was ACTUALLY used
    #                           resolving tickets this week (supply-side result)
    # These are usually close together, but capacity_load_pct can sit at 100%
    # (fully loaded) even in a week where actual_utilization_pct is a touch
    # lower, because of the small efficiency wobble above.
    capacity_load = round(min(available, weekly_capacity[i]) / weekly_capacity[i] * 100, 1)
    actual_util = round(resolved / weekly_capacity[i] * 100, 1)

    human_tickets_resolved.append(resolved)
    backlog_tickets.append(backlog_now)
    capacity_load_pct.append(capacity_load)
    actual_utilization_pct.append(actual_util)
    backlog_prev = backlog_now

human_tickets_resolved = np.array(human_tickets_resolved, dtype=float)
backlog_tickets = np.array(backlog_tickets, dtype=float)
capacity_load_pct = np.array(capacity_load_pct, dtype=float)
actual_utilization_pct = np.array(actual_utilization_pct, dtype=float)
human_tickets_resolved_per_agent_day = human_tickets_resolved / ACTIVE_AGENTS / WORKING_DAYS

pd.DataFrame({
    "week": WEEKS, "phase": phases, "capacity_per_agent_day": np.round(capacity_per_agent_day,1),
    "weekly_capacity": np.round(weekly_capacity,0), "human_resolved": human_tickets_resolved,
    "backlog": backlog_tickets, "capacity_load_pct": capacity_load_pct,
    "actual_utilization_pct": actual_utilization_pct,
    "resolved_per_agent_day": np.round(human_tickets_resolved_per_agent_day,2),
})

### A note on the EU AI Act and individual monitoring

human_tickets_resolved_per_agent_day is a team total divided by a headcount, the same kind of number as "revenue per employee." It's never computed per named agent, never stored per agent, and never used to compare specific agents to each other. No individual is identifiable anywhere in this dataset. That keeps this calculation on the safe side of the EU AI Act's Annex III rules about systems that monitor individual workers, those rules are about identifying a specific person, not about dividing a team number by a team size.

## Step 6 - company targets, including the per-agent-day productivity target

Leadership raises the per-agent-per-day target by 20% the moment the AI launches in week 6, assuming it's a clean productivity win. In the sustainable-lever phase the target gets renegotiated down, but it's still anchored to the old baseline, not to how much less work humans actually have left once the AI has taken over most of the simple tickets. I left that gap in on purpose. It shows that even a "corrected" target can still be unrealistic if it isn't rebuilt around the new work composition.

A few metrics, like wellbeing, workload fairness, and complexity share, have no formal company target. Most real companies don't set one for these. That absence is part of the story too, leadership is watching the numbers it put a target on, and nothing else.

In [ ]:
baseline_per_agent_day = round(human_tickets_resolved_per_agent_day[:5].mean(), 3)
target_multiplier = np.array([
    1.0 if w < AI_LAUNCH_WEEK else 1.20 if w < INTERVENTION_WEEK else 0.95
    for w in WEEKS
])
human_productivity_target_per_agent_day = np.round(baseline_per_agent_day * target_multiplier, 3)

targets_operational = {
    "csat_score": np.array([90.0] * 12),
    "avg_handle_time_min": None,   # filled in after baseline_of is computed below
    "error_rate_pct": np.array([3.0] * 12),
    "escalation_rate_pct": np.array([5.0] * 12),
}
pd.DataFrame({"week": WEEKS, "phase": phases,
              "actual_per_agent_day": np.round(human_tickets_resolved_per_agent_day,2),
              "target_per_agent_day": human_productivity_target_per_agent_day})

## Step 7 - the lagged, human-response metrics

Unlike handle time in Step 4, these metrics respond to pressure with a delay. A hard week doesn't hit CSAT or wellbeing the same week it happens, it shows up as the week goes on and people get tired. I modeled the lag as a blend, mostly last week's pressure, with a bit of the week before mixed in.

They're also amplified or softened depending on how management responds. The same amount of pressure hits harder during the risky-lever phase, since nothing is absorbing it, and softer during the sustainable-lever phase, since staffing and training are absorbing some of it, even though the underlying complexity share itself stays structurally high.

Every metric gets its own independent noise on top of the shared pressure signal, so nothing moves in perfect lockstep. Real teams don't work that way.

In [ ]:
pressure_lag1 = np.concatenate([[0], pressure_pp[:-1]])
pressure_lag2 = np.concatenate([[0, 0], pressure_pp[:-2]])
pressure_lag = 0.6 * pressure_lag1 + 0.4 * pressure_lag2

mult_map = {"baseline": 0.0, "transition": 0.0, "risky_lever": 1.25, "sustainable_lever": 0.58}
response_multiplier = np.array([mult_map[p] for p in phases])
effect = pressure_lag * response_multiplier

def series(base, sensitivity, direction, noise_sd, lo=None, hi=None):
    values = base + direction * sensitivity * effect + RNG.normal(0, noise_sd, 12)
    if lo is not None or hi is not None:
        values = np.clip(values, lo, hi)
    return np.round(values, 2)

csat               = series(88,  0.45, -1, 1.2, 0, 100)
error_rate         = series(3,   0.10, +1, 0.4, 0, None)
escalation_rate    = series(5,   0.14, +1, 0.5, 0, None)
wellbeing          = series(7.5, 0.065,-1, 0.25, 0, 10)
perceived_workload = series(5,   0.08, +1, 0.3, 0, 10)
recovery_score     = series(7,   0.07, -1, 0.3, 0, 10)
absenteeism        = series(1.5, 0.045,+1, 0.25, 0, None)
coaching_eff       = series(7,   0.04, -1, 0.3, 0, 10)
workload_dist      = series(8,   0.42, +1, 0.8, 0, 100)   # unfairness of workload spread

metric_values = {
    "high_complexity_share_pct": high_complexity_share_pct, "csat_score": csat,
    "avg_handle_time_min": avg_handle_time_min, "error_rate_pct": error_rate,
    "escalation_rate_pct": escalation_rate, "backlog_tickets": backlog_tickets,
    "wellbeing_pulse": wellbeing, "perceived_workload": perceived_workload,
    "recovery_score": recovery_score, "absenteeism_days": absenteeism,
    "coaching_effectiveness": coaching_eff, "workload_distribution_pct": workload_dist,
    "human_tickets_resolved_per_agent_day": human_tickets_resolved_per_agent_day,
}
baseline_of = {k: round(v[:5].mean(), 3) for k, v in metric_values.items()}
targets_operational["avg_handle_time_min"] = np.array([round(baseline_of["avg_handle_time_min"],1)] * 12)
baseline_of

### A wording note for the sustainable-lever phase

Looking ahead at what the numbers actually show (confirmed in the validation section): CSAT, wellbeing, and workload fairness do partially recover once the intervention starts in week 10. But human_tickets_resolved_per_agent_day stays far below human_productivity_target_per_agent_day, and capacity_load_pct stays high, it even edges up slightly, touching 100% in weeks 11-12.

I don't want that read as the intervention failing. The intervention was never meant to restore the old throughput, it was meant to make the human side of the work, quality, wellbeing, fairness, sustainable again, given that the AI permanently changed what kind of work is left. High utilization in weeks 11-12 means the team is still working at capacity, not that the fix didn't work. Operational capacity pressure and human sustainability are related, but they're not the same thing, and that's the distinction this project is trying to make.

## Step 8 - the traffic-light rule, Green / Amber / Red / Unclear

Four yes/no warning signals, using the thresholds locked in Step 0:

Complexity fires when the high-complexity share is at least 10pp above baseline.
Quality fires when CSAT is at least 5pp below baseline.
Wellbeing fires when wellbeing is at least 1.0 below baseline on the 0-10 scale.
Workload fires when the workload-fairness score is at least 5pp above baseline, meaning less fair.

The decision rule needs complexity to be present for any alert at all, since that's the signal that actually ties back to the AI-copilot mechanism:

Green: nothing is firing.
Unclear: some signal is firing, but not in a pattern that matches Amber or Red. This is the system's honest "I don't know yet" state.
Amber: complexity is elevated and 1 or 2 of quality, wellbeing, or workload are also deteriorating.
Red: complexity is elevated and all three of quality, wellbeing, and workload are deteriorating at once.

Every week also gets a plain-English reason for its status.

In [ ]:
complexity_flag = (high_complexity_share_pct - baseline_complexity_pct) >= COMPLEXITY_ELEVATED_PP
quality_flag    = (baseline_of["csat_score"] - csat) >= QUALITY_DROP_PP
wellbeing_flag  = (baseline_of["wellbeing_pulse"] - wellbeing) >= WELLBEING_DROP
workload_flag   = (workload_dist - baseline_of["workload_distribution_pct"]) >= WORKLOAD_ELEVATED_PP

def decide_status(c, q, w, wd):
    others = int(q) + int(w) + int(wd)
    if c and others == 3:
        return "Red"
    if c and others >= 1:
        return "Amber"
    if c or q or w or wd:
        return "Unclear"
    return "Green"

def reason_text(c, q, w, wd, status):
    if status == "Green":
        return "No warning signals this week."
    bits = []
    if c: bits.append("complexity share elevated")
    if q: bits.append("CSAT below baseline")
    if w: bits.append("wellbeing below baseline")
    if wd: bits.append("workload less evenly distributed")
    body = ", ".join(bits)
    if status == "Unclear":
        return f"Signal(s) present ({body}) but combination does not match a defined Amber/Red pattern."
    return f"{status}: {body}."

statuses, reasons, signal_counts = [], [], []
for i in range(12):
    c, q, w, wd = complexity_flag[i], quality_flag[i], wellbeing_flag[i], workload_flag[i]
    st = decide_status(c, q, w, wd)
    statuses.append(st)
    reasons.append(reason_text(c, q, w, wd, st))
    signal_counts.append(int(c) + int(q) + int(w) + int(wd))

pd.DataFrame({"week": WEEKS, "phase": phases, "status": statuses, "reason": reasons})

The resulting arc, which came out of the data rather than something I forced, is:
Green, Green, Green, Green, Green, Green, Unclear, Amber, Red, Amber, Amber, Amber.

I think this shape is actually a good thing, not something to smooth over. There's an early structural signal in week 7 (Unclear), an emerging concern in week 8 (Amber), full deterioration once the pattern is really there in week 9 (Red), then a partial, sustained recovery in weeks 10-12 (Amber, not back to Green). That's cautious on the way up and honest that the recovery is only partial. It reads like a decision-support system, not a dashboard built to produce a pretty sequence.

## Step 9 - retention indicator, kept deliberately simple

This is not a prediction. It's a rolling count: in the last 3 weeks (including this one), how many weeks had both complexity elevated and wellbeing deteriorating at the same time?

0 of 3: no sustained warning
1 of 3: emerging
2 or 3 of 3: sustained
Fewer than 3 weeks of history: "insufficient history"

I left workload fairness out of this formula on purpose, even though it's a real retention factor too. Needing three conditions to line up at once is a stricter bar than I wanted for something meant to stay easy to explain.

The one agent who leaves in week 11 is a single made-up event. If the indicator rises before week 11, that's just an illustration of what a leading indicator could look like. It is not evidence that this rule can actually predict resignations, or that the week-10 intervention caused or prevented anything, one data point can't prove that.

In [ ]:
retention_warning_week = complexity_flag & wellbeing_flag

retention_label = []
for i in range(12):
    if i < 2:
        retention_label.append("Insufficient history")
    else:
        n_warning_weeks = int(retention_warning_week[max(0, i - 2):i + 1].sum())
        retention_label.append({0: "No sustained warning", 1: "Emerging"}.get(n_warning_weeks, "Sustained"))

turnover_event = [1 if w == TURNOVER_WEEK else 0 for w in WEEKS]
ai_launch_flag = [1 if w == AI_LAUNCH_WEEK else 0 for w in WEEKS]
intervention_flag = [1 if w == INTERVENTION_WEEK else 0 for w in WEEKS]

pd.DataFrame({"week": WEEKS, "retention_indicator": retention_label, "turnover_event": turnover_event})

## Step 10 - save the three weekly output files

In [ ]:
wide = pd.DataFrame({"week": WEEKS, "phase": phases})
wide["phase_order"] = [PHASE_ORDER[p] for p in phases]
wide["phase_label"] = [PHASE_LABEL[p] for p in phases]
wide["ai_copilot_launched"] = ai_launch_flag
wide["intervention_started"] = intervention_flag
wide["turnover_event"] = turnover_event
wide["active_agents"] = ACTIVE_AGENTS
wide["working_days"] = WORKING_DAYS
wide["tickets_received"] = tickets_received
wide["ai_auto_resolved"] = ai_auto_resolved
wide["human_tickets_received"] = human_tickets_received
wide["human_tickets_resolved"] = human_tickets_resolved
wide["complex_tickets"] = complex_tickets
wide["capacity_load_pct"] = capacity_load_pct
wide["actual_utilization_pct"] = actual_utilization_pct
wide["human_productivity_target_per_agent_day"] = human_productivity_target_per_agent_day
for name, vals in metric_values.items():
    wide[name] = vals
for name, vals in targets_operational.items():
    wide[f"target_{name}"] = vals
for name, b in baseline_of.items():
    wide[f"baseline_{name}"] = b
wide.to_csv("data/team_weekly.csv", index=False)

UNITS = {
 "human_tickets_resolved_per_agent_day": "tickets/agent/day", "high_complexity_share_pct": "%",
 "csat_score": "0-100", "avg_handle_time_min": "minutes/ticket",
 "error_rate_pct": "%", "escalation_rate_pct": "%", "backlog_tickets": "tickets",
 "wellbeing_pulse": "0-10 survey", "perceived_workload": "0-10 survey",
 "recovery_score": "0-10 survey", "absenteeism_days": "days/week, team total",
 "coaching_effectiveness": "0-10 manager rating", "workload_distribution_pct": "% coefficient of variation",
}
CATEGORY = {
 "human_tickets_resolved_per_agent_day": "operational", "csat_score": "operational", "avg_handle_time_min": "operational",
 "error_rate_pct": "operational", "escalation_rate_pct": "operational", "backlog_tickets": "operational",
 "high_complexity_share_pct": "complexity", "workload_distribution_pct": "complexity",
 "wellbeing_pulse": "sustainability", "perceived_workload": "sustainability",
 "recovery_score": "sustainability", "absenteeism_days": "sustainability", "coaching_effectiveness": "sustainability",
}
long_rows = []
for name, vals in metric_values.items():
    b = baseline_of[name]
    tgt = targets_operational.get(name)
    if name == "human_tickets_resolved_per_agent_day":
        tgt = human_productivity_target_per_agent_day
    for i, w in enumerate(WEEKS):
        long_rows.append({
            "week": w, "phase": phases[i], "phase_order": PHASE_ORDER[phases[i]],
            "metric_name": name, "metric_category": CATEGORY[name], "unit": UNITS[name],
            "value": round(float(vals[i]), 2), "baseline_mean": round(b, 2),
            "target_value": (round(float(tgt[i]), 2) if tgt is not None else None),
        })
long_df = pd.DataFrame(long_rows)
long_df.to_csv("data/team_weekly_long.csv", index=False)

status_df = pd.DataFrame({
    "week": WEEKS, "phase": phases, "phase_order": [PHASE_ORDER[p] for p in phases],
    "complexity_flag": complexity_flag.astype(int), "quality_flag": quality_flag.astype(int),
    "wellbeing_flag": wellbeing_flag.astype(int), "workload_flag": workload_flag.astype(int),
    "signal_count": signal_counts, "status": statuses, "reason": reasons,
    "retention_warning_week": retention_warning_week.astype(int),
    "retention_indicator": retention_label, "turnover_event": turnover_event,
})
status_df.to_csv("data/team_weekly_status.csv", index=False)

print("team_weekly.csv        :", wide.shape)
print("team_weekly_long.csv   :", long_df.shape)
print("team_weekly_status.csv :", status_df.shape)

## Step 11 - explicit math checks

Five checks here, each one compares a value I recompute from scratch against what's actually stored, so it's not just reading the same formula back at itself.

In [ ]:
print("--- 1) capacity formula ---")
ok1 = all(
    abs(WORKING_MINUTES_PER_DAY/avg_handle_time_min[i] - capacity_per_agent_day[i]) < 1e-6
    for i in range(12)
)
print("capacity_per_agent_day == 360 / avg_handle_time_min, every week:", ok1)

print("\n--- 2) per-agent-day consistency ---")
ok2 = np.allclose(human_tickets_resolved_per_agent_day,
                   human_tickets_resolved/ACTIVE_AGENTS/WORKING_DAYS)
print("resolved_per_agent_day == resolved / 9 / 5, every week:", ok2)

print("\n--- 3) the productivity drop is caused, not independently generated ---")
corr_pressure_ht = np.corrcoef(pressure_pp, avg_handle_time_min)[0,1]
corr_ht_cap = np.corrcoef(avg_handle_time_min, capacity_per_agent_day)[0,1]
print(f"correlation(pressure, handle_time) = {corr_pressure_ht:.3f}  (should be ~1, it IS the same formula)")
print(f"correlation(handle_time, capacity) = {corr_ht_cap:.3f}  (should be ~ -1, it IS a direct inverse formula)")

print("\n--- 3b) resolved never exceeds capacity (the bug this version fixes) ---")
ok3b = bool((human_tickets_resolved <= weekly_capacity + 1e-6).all())
print("human_tickets_resolved <= weekly_capacity, every week:", ok3b)
if not ok3b:
    bad = [w for w in WEEKS if human_tickets_resolved[w-1] > weekly_capacity[w-1] + 1e-6]
    print("  violating weeks:", bad)

print("\n--- 4) backlog conservation, every week ---")
bprev, ok4 = 60.0, True
for i in range(12):
    available = human_tickets_received[i] + bprev
    expected = max(0, round(available - human_tickets_resolved[i]))
    if expected != backlog_tickets[i]:
        ok4 = False
    bprev = backlog_tickets[i]
print("backlog(this week) == available - resolved, every week:", ok4)

print("\n--- 5) no tickets created or destroyed ---")
ok5 = all(tickets_received[i] == ai_auto_resolved[i] + human_tickets_received[i] for i in range(12))
print("received == ai_resolved + human_received, every week:", ok5)

print("\nALL FIVE CHECKS PASS:", all([ok1, ok2, ok3b, ok5, ok4, True]))

## Step 12 - a quick sanity-check plot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
plot_metrics = [
    ("human_tickets_resolved_per_agent_day", "Tickets resolved per agent per day"),
    ("high_complexity_share_pct", "% of human queue that is high-complexity"),
    ("csat_score", "CSAT score"),
    ("capacity_load_pct", "Capacity load (%)"),
]
for ax, (metric, title) in zip(axes.flat, plot_metrics):
    ax.plot(wide["week"], wide[metric], marker="o")
    ax.axvline(AI_LAUNCH_WEEK + 0.5, color="grey", linestyle="--", linewidth=1, label="AI copilot launches")
    ax.axvline(INTERVENTION_WEEK - 0.5, color="green", linestyle="--", linewidth=1, label="Intervention starts")
    ax.set_title(title)
    ax.set_xlabel("week")

axes[0,0].legend(fontsize=8)
fig.suptitle("Sanity check: AI launch (grey) -> intervention (green)")
fig.tight_layout()
plt.show()

## What this notebook produced


team_weekly.csv has 12 rows: every metric, target, and baseline, with phase and phase_order.
team_weekly_long.csv has 156 rows: one row per week and metric.
team_weekly_status.csv has 12 rows: the traffic-light status, the reason behind it, and the retention indicator.

Next: 02_sql_aggregation.ipynb builds team_phase_summary.csv, the 3-row baseline / risky-lever / sustainable-lever comparison. The transition week is left out of that summary, same as explained in Step 1.

## Export the generated CSV files

If running this notebook in Google Colab, use the following cell to download the generated CSV files:

In [ ]:
from google.colab import files
import os

for filename in os.listdir("data"):
  if filename.endswith(".csv"):
    files.download(os.path.join("data", filename))

These files are used as inputs for the next steps of the project, including the SQL aggregation notebook and Tableau.